# analyzing: the token `Matcher`
The token `Matcher` is very similar to the `PhraseMatcher` from two sections before. The difference is that this `Matcher` is allows for more variation, so we can capture different forms of the same basic pattern. For example, we could get definitions of gender (and sex, and sexuality) that use different words (like "means" or "includes") or kinds of punctuation (like single or double quotes) in the defintition.

The token `Matcher` works by writing a pattern of tokens that we define using the token attributes. We can, for example, leverage the work we did with the `EntityRuler` in the previous section to help create our token `Matcher`.

First, we will import the matcher to create a matcher object. Then, we will write patterns and save them. After that, we add our new patterns to the matcher. Finally, we will run the matcher on our document. 

Let's take it one step at a time. 

In [1]:
# import our matcher class
import spacy
from spacy.matcher import Matcher

nlp = spacy.load("en_core_web_sm")

with open ('./out/clean.txt', 'r') as f:
    text = f.read()

This is the basic format of the `Matcher`. We will add much more detail to this format later on, but it's a good idea to get a sense of how it's structured now, using JSON key-value pairs. 

Also, it draws the token attributes from this page: https://spacy.io/api/matcher

In [11]:
pattern_format = [
    {
        'LOWER': 'term'
    },
    {
        'IS_PUNCT': True
    },
    {
        'LOWER': 'gender'
    },
    {
        'IS_PUNCT': True
    },
    {
        'LOWER': 'means'
    }
]

We want to capture not just "gender," but "sex" and "sexuality," as well as other synonyms for these terms. That's where the custom entities from the last section will become useful. 

In [12]:
ruler = nlp.add_pipe("entity_ruler", after="ner")

patterns = [
                {"label": "GENDER", "pattern": 'gender'},
                {"label": "GENDER", "pattern": 'trans'},
                {"label": "GENDER", "pattern": 'nonbinary'},
                {"label": "GENDER", "pattern": 'male'},
                {"label": "GENDER", "pattern": 'female'},
                {"label": "SEX", "pattern": 'sex'},
                {"label": "SEX", "pattern": 'biological'},
                {"label": "SEXUALITY", "pattern": 'sexuality'},
                {"label": "SEXUALITY", "pattern": 'orientation'},
                {"label": "SEXUALITY", "pattern": 'queer'},
                {"label": "IDENTITY", "pattern": 'LGBTQ'},
                {"label": "IDENTITY", "pattern": 'LGBT'},
                {"label": "IDENTITY", "pattern": 'LGBTQIA+'},
                {"label": "IDENTITY", "pattern": 'queer'}
            ]

ruler.add_patterns(patterns)
doc = nlp(text)

ValueError: [E007] 'entity_ruler' already exists in pipeline. Existing names: ['tok2vec', 'tagger', 'parser', 'senter', 'attribute_ruler', 'lemmatizer', 'ner', 'entity_ruler']

We want to capture a specific pattern where gender is being defined. We'd want a phrase that captures text like: "the term gender means", and to also get variations on the punctuation and/or terminologies in that text. For example, we want to get instances where they use both single and double quotes. 

In [4]:
pattern = [
      {

          # "LOWER" indicates the word we want in lowercase form
          # "OP" is an option we can define, here to indicate that a
          # pattern can match, but doesn't have to match. 
          'LOWER': 'term', 'OP': '*' # allows the pattern to match zero or more times
      }, 
      {
          # catches punctuation if it appears zero or more times
          'IS_PUNCT': True, 'OP': '*'
      },
      {
          # specifying the entity type, which can be one of our three
          # custom entities
          "ENT_TYPE": {
              'IN': [
                  'GENDER', 'SEX', 'SEXUALITY'
              ]
          }
      },
      {'OP': '?'}, # catches a "wild card" if it appears zero or one time.
      {
          'IS_PUNCT': True, 'OP': '*' # allows the pattern to match zero or more times
      },
      {
          # getting the lowercase word of any of the following terms
          'LOWER': {
              'IN': [
                  'means', 'signifies', 'includes'
              ]
          }
      }
  ]

Now we can configure the `Matcher`. First, create the matcher, then add our pattern to the matcher, and finally run the mather on our doc. 

In [13]:
# use matcher class to create a matcher object
matcher = Matcher(nlp.vocab)

# add pattern to matcher
matcher.add('definition', [pattern])

# run matcher over doc
matches = matcher(doc)

In [14]:
# how many matches did we get?

len(matches)

62

Let's see the actual text. 

In [15]:
for match_id, start, end in matches[:10]:
    string_id = nlp.vocab.strings[match_id]  # Get string representation
    span = doc[start:end]  # The matched span
    print(string_id, start, end, span.text)
    print(doc[start].sent)

definition 918 921 sex' means
Biological sex.--The term `biological sex' means the biological indication of male or female in the context of reproductive potential or capacity, such as sex chromosomes, naturally occurring sex hormones, gonads, and non-ambiguous internal and external genitalia present at birth, without regard to an individual's psychological, chosen, or subjective experience of gender.
definition 917 921 biological sex' means
Biological sex.--The term `biological sex' means the biological indication of male or female in the context of reproductive potential or capacity, such as sex chromosomes, naturally occurring sex hormones, gonads, and non-ambiguous internal and external genitalia present at birth, without regard to an individual's psychological, chosen, or subjective experience of gender.
definition 1048 1051 gender' means
Gender.--The term `gender' means the psychological, behavioral, social, and cultural aspects of being male or female.
definition 1073 1077 gende

Due to the versatility of the token `Matcher`, we can catch instances like gender dysphoria'' means and orientation includes, which goes beyond what we were able to do with the `PhraseMatcher`. Pretty cool, right?

Next step is to save our data as a plain text file, so we can review it later.

We will include just the matched phrase and the full sentence from which it occurs. 

In [16]:
with open('matcher_defs.txt', 'w') as f:
    for match_id, start, end in matches:
        string_id = nlp.vocab.strings[match_id]  # Get string representation
        span = doc[start:end]  # The matched span
        f.write(str(span.text))
        f.write(str(doc[start].sent))
        f.write(str('\n'))
        f.write(str('\n'))

That's all, folks!